# Notebook 06 - Coupled Piping-Structural Systems and Design Evaluation

This lesson extends the pipe-only workflow into a coupled pipe-rack model and scores one explicit support layout from Code_Aster-backed results. No optimization is run in this lesson.

You will do four things:

1. Build a portal frame and pipe crossing in one `TubaModel`.
2. Add a frictional rest support at the actual crossing node.
3. Score the design from Code_Aster-backed results.
4. Distinguish evaluation of one layout from a repeated-solve optimization search.


## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pyvista as pv

# Setup repo root for path import
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuba import Model
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results
from tuba.plotting import plots

# Enable interactive notebook rendering
# Defaults to zoomable embedded HTML locally; set TUBA_NOTEBOOK_BACKEND=client or static to override.
from tuba.plotting.notebook import configure_notebook_backend
JUPYTER_BACKEND = configure_notebook_backend()

## 2. Building the Coupled Portal Frame + Piping Model

We'll build a structural portal frame at $X = 5.0$ meters:
- Two vertical columns modeled as beams using a box section profile (`ColBox`).
- One horizontal girder connecting the column tops modeled as an I-beam (`HE200B`).
- A piping system crossing directly over the girder at $Y = 3.1143$ meters with a frictional rest support.

The pipe run is still built with `PipingBuilder`, but the steel portal frame is easier to show with explicit node and beam calls. `model.add_node([x, y, z])` returns a generated node id. `model.add_element(..., n1=start_node, n2=end_node, ...)` connects two node ids; the beam vector is `coords(n2) - coords(n1)`.

In [ ]:
model = Model("StructuralDemo", standard="ASME_B31.3")

model.add_material(
    "S235JR",
    E=2.1e11,
    nu=0.3,
    alpha=1.2e-5,
    rho=7850.0,
    allowable_stress={20.0: 137e6, 200.0: 120e6},
)
model.add_pipe_section("4inch_sch40", OD=0.1143, WT=0.00602)
model.add_rectangular_section("ColBox", height_y=0.2, height_z=0.2, thickness_y=0.008, thickness_z=0.008)
model.add_ibeam_section("HE200B_Girder", "HE200B")

c1_base = model.add_node([5.0, 0.0, -1.0])
c1_top = model.add_node([5.0, 3.0, -1.0])
model.add_element(id="col_1", type="beam", n1=c1_base, n2=c1_top, section="ColBox", material="S235JR")
model.add_support(c1_base, "anchor")

c2_base = model.add_node([5.0, 0.0, 1.0])
c2_top = model.add_node([5.0, 3.0, 1.0])
model.add_element(id="col_2", type="beam", n1=c2_base, n2=c2_top, section="ColBox", material="S235JR")
model.add_support(c2_base, "anchor")

model.add_element(id="girder", type="beam", n1=c1_top, n2=c2_top, section="HE200B_Girder", material="S235JR")

frame_nodes = {
    "c1_base": c1_base,
    "c1_top": c1_top,
    "c2_base": c2_base,
    "c2_top": c2_top,
}
frame_elements = ("col_1", "col_2", "girder")

print("Frame node coordinates:")
for label, node_id in frame_nodes.items():
    print(f"  {label:<7} -> {node_id}: {model.nodes[node_id].coords.tolist()}")

print("Frame element endpoint vectors:")
for element_id in frame_elements:
    elem = model.get_element(element_id)
    start = model.nodes[elem.n1].coords
    end = model.nodes[elem.n2].coords
    print(f"  {elem.id}: n1={elem.n1}, n2={elem.n2}, vector={(end - start).tolist()}")

with model.pipe(section="4inch_sch40", material="S235JR") as b:
    b.start([0, 3.1143, 0], support="anchor")
    b.run(5.0)
    b.run(5.0)
    b.end(support="anchor")

pipe_crossing_node = next(
    nid for nid, node in model.nodes.items()
    if np.allclose(node.coords, [5.0, 3.1143, 0.0])
)
print(f"Crossing node over the girder: {pipe_crossing_node}")
print(f"Model has {len(model.nodes)} nodes and {len(model.elements)} elements.")


Attach the frictional rest to the computed crossing node directly above the structural girder.


In [ ]:
model.add_support(pipe_crossing_node, type="rest", friction_coefficient=0.3)
model.define_load_case("Operating_Hot", gravity=True, pressure=2.5e6, temperature=220.0, ref_temperature=20.0)
print(f"Rest support attached to {pipe_crossing_node}.")


### Visualizing the Coupled Structural-Piping Rack

In [ ]:
from tuba.plotting.pipeline import build_3d_mesh_from_model

# True section geometry — the box columns and I-beam girder show their real
# profiles straight from the model's sections, not uniform tubes.
tubes = build_3d_mesh_from_model(model)

p = pv.Plotter()
p.set_background("#1a1a2e")
p.add_mesh(tubes, color="#5c6b73")
plots._add_supports_to_plotter(p, model, scale=0.18)
p.show(jupyter_backend=JUPYTER_BACKEND)

## 3. Code_Aster Result Review

Review the explicit support layout from the loaded Code_Aster artifacts. The summary reports the persisted result state, ASME B31.3 compliance, and the current operating-state clash check without assigning a synthetic design score.

In [ ]:
from tuba.analysis.states import create_cold_geometry_state, create_operating_geometry_state
from tuba.clash import ClashEngine
from tuba.compliance.asme_b313 import ASMEB313Evaluator

CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
# VS Code/Jupyter review defaults to committed real Code_Aster artifacts; set True only after the runtime doctor passes.
RUN_CODE_ASTER = False
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "structural_operating_hot"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating_Hot",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact
result_state = code_aster_artifact.result_state

if code_aster_run.ran_solver:
    print("Code_Aster solver executed for this notebook run.")
compliance_report = ASMEB313Evaluator().evaluate(model, results)
cold_state = create_cold_geometry_state(model)
operating_state = create_operating_geometry_state(model=model, result_state=result_state)
operating_clashes = ClashEngine().check_operating_state(
    model,
    cold_state=cold_state,
    operating_state=operating_state,
    result_state=result_state,
    analysis_mesh=code_aster_artifact.analysis_mesh,
)
max_displacement_m = max(
    (np.linalg.norm(displacement[:3]) for displacement in result_state.node_displacements.values()),
    default=0.0,
)
max_reaction_n = max(
    (np.linalg.norm(reaction[:3]) for reaction in result_state.node_reactions.values()),
    default=0.0,
)
print("Code_Aster-backed review summary:")
print(f"  Result state: {result_state.id} ({result_state.load_case})")
print(f"  ASME B31.3 overall pass: {compliance_report.overall_pass}")
print(f"  Worst sustained ratio: {compliance_report.worst_sustained_ratio:.3f}")
print(f"  Worst expansion ratio: {compliance_report.worst_expansion_ratio:.3f}")
print(f"  Maximum displacement: {max_displacement_m * 1000.0:.2f} mm")
print(f"  Maximum reaction: {max_reaction_n:.1f} N")
print(f"  Operating-state clashes: {len(operating_clashes)}")

## 4. Review Boundary

This notebook reviews one explicit support layout. Its displayed engineering values remain tied to the loaded Code_Aster result artifacts.


In [ ]:
print("The explicit supports and loaded Code_Aster result artifacts remain authoritative.")

## Key Takeaways

- Mixed structural and piping elements can live in one `TubaModel`.
- The review cell reports direct Code_Aster-backed result, compliance, and clash summaries.
- The explicit support layout is reviewed without a synthetic optimization score.

Next: `07_bim_data_exchange.ipynb` exports the model and solver properties into data-exchange formats.
